In [1]:
import torch
import torch.nn as nn

**Mean Aggregator**

In [ ]:
class MeanAggregator(nn):
  def __init__(self, input_dim, output_dim, neigh_input_dim = None, 
               bias = False, act = nn.ReLU, concat = False):
    super(MeanAggregator, self).__init__()

    self.bias = bias
    self.act = act
    self.concat = concat
    self.input_dim = input_dim
    self.output_dim = output_dim

    if neigh_input_dim is None:
      neigh_input_dim = input_dim
    
    self.neigh_weights = nn.parameter(torch.empty(neigh_input_dim, output_dim))
    nn.init.xavier_uniform_(self.neigh_weights)

    self.self_weights = nn.parameter(torch.empty(input_dim, output_dim))
    nn.init.xavier_uniform_(self.self_weights)

    if self.bias:
      self.bias_var = nn.parameter(torch.empty(output_dim, 1))
      nn.init.zeros_(self.bias)


  def __call__(self, inputs):
    self_vecs, neigh_vecs = inputs

    neigh_means = torch.mean(neigh_vecs, dim=1)

    neigh_transformed = torch.matmul(neigh_means, self.neigh_weights)
    self_transformed = torch.matmul(self_vecs, self.self_weights)

    if not self.concat:
      ouput = torch.add(neigh_transformed, self_transformed)
    else:
      output = torch.cat((neigh_transformed, self_transformed), dim = 1)

    if self.bias:
      output += self.bias_var

    return self.act(output)


**GCN Aggregator**

In [ ]:
class GCNAggregator(nn):
  def __init__(self, input_dim, output_dim, neigh_input_dim = None, 
               bias = False, act = nn.ReLU, concat = False):
    super(GCNAggregator, self).__init__()

    self.bias = bias
    self.act = act
    self.concat = concat
    self.input_dim = input_dim
    self.output_dim = output_dim

    if neigh_input_dim is None:
      neigh_input_dim = input_dim
    
    self.weights = nn.parameter(torch.empty(neigh_input_dim, output_dim))
    nn.init.xavier_uniform_(self.neigh_weights)

    if self.bias:
      self.bias_var = nn.parameter(torch.empty(output_dim, 1))
      nn.init.zeros_(self.bias)

  def __call__(self, inputs):
    self_vecs , neigh_vecs = inputs

    means = torch.cat([neigh_vecs, self_vecs.unsqueeze(1)], dim=1).mean(dim=1)

    output = torch.matmul(means, self.weights)

    if self.bias:
      output += self.bias_var

    return self.act(output)

**MaxPool Aggregator**

In [ ]:
def MaxPoolAggregator(nn):
  def __init__(self, input_dim, output_dim, neigh_input_dim = None, hidden_dim = 256,
                bias = False, act = nn.ReLU, concat = False):
    super(MeanAggregator, self).__init__()

    self.bias = bias
    self.act = act
    self.concat = concat
    self.input_dim = input_dim
    self.output_dim = output_dim

    if neigh_input_dim is None:
      neigh_input_dim = input_dim
    
    self.mlp_layers = []
    self.mlp_layers.append(
      nn.Sequential(
        nn.Liear(neigh_input_dim, hidden_dim),
        nn.ReLU()
      )
    )

    self.neigh_weights = nn.parameter(torch.empty(neigh_input_dim, output_dim))
    nn.init.xavier_uniform_(self.neigh_weights)

    self.self_weights = nn.parameter(torch.empty(input_dim, output_dim))
    nn.init.xavier_uniform_(self.self_weights)

    if self.bias:
      self.bias_var = nn.parameter(torch.empty(output_dim, 1))
      nn.init.zeros_(self.bias)


  def __call__(self, inputs):
    self_vecs, neigh_vecs = inputs
    
    
    neigh_means = torch.mean(neigh_vecs, dim=1)

    neigh_transformed = torch.matmul(neigh_means, self.neigh_weights)
    self_transformed = torch.matmul(self_vecs, self.self_weights)

    if not self.concat:
      ouput = torch.add(neigh_transformed, self_transformed)
    else:
      output = torch.cat((neigh_transformed, self_transformed), dim = 1)

    if self.bias:
      output += self.bias_var

    return self.act(output)